# Behavioral Master Signal — Interactive Demo

> **Prototype on synthetic data.**  Results do not claim real-world predictive validity.

This notebook walks through the full pipeline interactively:
1. Generate synthetic data
2. Attach labels
3. Build sub-signals & heuristic score
4. Train ML models
5. Evaluate
6. Explain
7. Visualise


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))  # repo root

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
%matplotlib inline

from src.synthetic_data import generate_panel
from src.labels import attach_labels
from src.features import build_sub_signals, build_model_features
from src.scoring import attach_heuristic_scores
from src.modeling import train_models, save_models
from src.evaluation import evaluate_all, decile_table
from src.explainability import run_explainability
from src import config

print("Imports OK")

## Step 1 — Generate Synthetic Panel

Simulates 300 clients × 48 months with realistic market dynamics.

In [ ]:
panel = generate_panel()
print(f"Panel shape: {panel.shape}")
panel.head(3)

## Step 2 — Attach Labels

Labels are generated from latent probabilities — never the reverse.

In [ ]:
panel = attach_labels(panel)
print("Label rates:")
print(panel[["buy_3m", "redeem_3m"]].mean().to_string())

## Step 3 — Build Sub-Signals & Heuristic Score

In [ ]:
panel = build_sub_signals(panel)
panel = attach_heuristic_scores(panel)
panel[config.SUB_SIGNAL_COLS + ["master_buy_score_heuristic", "master_redeem_score_heuristic"]].describe().round(3)

## Step 4 — Time-Aware Split & Model Training

In [ ]:
from src.utils import time_split
train_df, valid_df, test_df = time_split(panel)
print(f"Train: {len(train_df):,}  Valid: {len(valid_df):,}  Test: {len(test_df):,}")

X = build_model_features(panel)
feature_cols = X.columns.tolist()
panel_ml = panel.copy()
for col in feature_cols:
    if col not in panel_ml.columns:
        panel_ml[col] = X[col]

models = train_models(panel_ml, feature_cols)
print("Models trained.")

## Step 5 — Evaluation

In [ ]:
results_df = evaluate_all(models, panel_ml)
results_df.round(3)

## Step 6 — Explainability & Narratives

In [ ]:
explain = run_explainability(models, panel_ml, n_shap_rows=200)
print("
--- Buy model: top 5 SHAP features ---")
print(explain["buy_3m"]["shap_feature_importance"].head(5).to_string(index=False))

print("
--- Example narratives (buy_3m) ---")
for k, v in explain["buy_3m"]["example_narratives"].items():
    print(f"  [{k}] {v}")

## Step 7 — Visualisations

In [ ]:
from src.plotting import (
    plot_signal_distributions, plot_score_distributions,
    plot_model_performance, plot_lift_curves,
    plot_feature_importance, plot_client_timeline
)

fig1 = plot_signal_distributions(panel)
plt.show()

fig2 = plot_model_performance(results_df)
plt.show()

fig3 = plot_lift_curves(models, panel_ml)
plt.show()

fig4 = plot_feature_importance(explain)
plt.show()

fig5 = plot_client_timeline(panel)
plt.show()

## Decile Lift Table (GBT — redeem_3m — Test Set)

In [ ]:
from src.utils import time_split
from src.evaluation import decile_table

_, _, test_df = time_split(panel_ml)
feat = models["redeem_3m"]["feature_cols"]
X_te = test_df[feat].astype(float)
y_te = test_df["redeem_3m"].astype(int).values
probs = models["redeem_3m"]["gbt"].predict_proba(X_te)[:, 1]

dt = decile_table(y_te, probs)
dt.round(3)

---
**Disclaimer:** All results above are on synthetic data. They do not claim real-world predictive validity.